# Data Prep 2
The purpose of this notebook is to transform the data in preparation for the algos. This involves scaling and transforming the numerical data. In addition there will be feature engineering. This notebook focuses on the second wind dataset

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from pathlib import Path
import sys
import joblib
from datetime import datetime, date, timedelta

In [2]:
# set up relative imports
project_folder = Path.cwd().parent.parent
sys.path.append(str(project_folder))

In [8]:
from modeling.utilities.data_prep import f_to_c, convert_to_cyclical, create_split_dfs

In [4]:
df = pd.read_parquet(r'..\data\clean-merged-water-weather-wind-data.parquet')

In [5]:
df.head(1)

,date,WSPD,WDIR,TMAX,TMIN,PRCP,BOT_TEMP_C,SURF_TEMP_C,month,day,year
0,2005-01-01,2.26087,185.0,61.0,47.0,0.0,15.4,15.3,1,1,2005


In [6]:
# convert all temps to celcius
df['tmax_c'] = f_to_c(df.TMAX)
df['tmin_c'] = f_to_c(df.TMIN)

# add the sin and cos cyclical columns for day and month
df['sin_month'] = convert_to_cyclical(np.sin, df.month, 12)
df['cos_month'] = convert_to_cyclical(np.cos, df.month, 12)

# for day we are just assuming every month to 31 
df['sin_day'] = convert_to_cyclical(np.sin, df.day, 31)
df['cos_day'] = convert_to_cyclical(np.cos, df.day, 31)

# sin and cos of the wind
df['sin_wdir'] = np.sin(2*np.pi*df.WDIR / 360)
df['cos_wdir'] = np.cos(2*np.pi*df.WDIR / 360)

# east and north wind components
df['wind_u'] = df.WSPD * df.sin_wdir
df['wind_v'] = df.WSPD * df.cos_wdir

This dataset has known lapses in time frames, so we will have to be careful with creating rolling averages and the target variables. Essentially the dataset will have to be split into chunks where it is continuous.

In [11]:
slice_dfs = create_split_dfs(df)

In [12]:
len(df)

5698

In [13]:
np.sum([len(x) for x in slice_dfs])

np.int64(5698)

In [ ]:
# above confirms that the df was sliced up properly

In [14]:
def create_cols(df):
    """
    this function is for creating extra columns per df chunk
    """
    df = df.copy()
    # surface temp
    df['past_10_day_ave_temp'] = df.SURF_TEMP_C.rolling(window=10,min_periods=1).mean()
    df['past_10_day_std_temp'] = df.SURF_TEMP_C.rolling(window=10,min_periods=1).std()

    df['wind_u_3day_ave'] = df.wind_u.rolling(window=3,min_periods=1).mean()
    df['wind_v_3day_ave'] = df.wind_v.rolling(window=3,min_periods=1).mean()

    df['target'] = df.SURF_TEMP_C.shift(1)

    df = df.dropna(subset='target')
    
    return df



In [15]:
updated_dfs = [create_cols(x) for x in slice_dfs]

In [16]:
updated_dfs[0].head(2)

,date,WSPD,WDIR,TMAX,TMIN,PRCP,BOT_TEMP_C,SURF_TEMP_C,month,day,...,cos_day,sin_wdir,cos_wdir,wind_u,wind_v,past_10_day_ave_temp,past_10_day_std_temp,wind_u_3day_ave,wind_v_3day_ave,target
1,2005-01-02,3.333333,117.391304,59.0,47.0,0.00,15.1,15.1,1,2,...,0.918958,0.887885,-0.460065,2.959617,-1.533550,15.200000,0.141421,1.381285,-1.892908,15.3
2,2005-01-03,3.458333,118.750000,60.0,52.0,1.08,15.0,15.0,1,3,...,0.820763,0.876727,-0.480989,3.032013,-1.663419,15.133333,0.152753,1.931528,-1.816412,15.1


In [17]:
all_df = pd.concat(updated_dfs)

In [18]:
all_df.shape

(5673, 26)

#### split, scale, and save the data

In [19]:
# drop the date column
all_df.drop(columns=['date'], inplace=True)

In [20]:
train_dfs = []
test_dfs = []
val_dfs = []
window = 255

for start in range(0, len(all_df), window):
    end = min(start + window, len(all_df))

    slice_df = all_df[start: end]
    
    n = len(slice_df)
    train_end = int(.7 * n)
    val_end = int(.85*n)

    train_dfs.append(slice_df[:train_end])
    val_dfs.append(slice_df[train_end:val_end])
    test_dfs.append(slice_df[val_end:])


In [21]:
train_df = pd.concat(train_dfs)
test_df = pd.concat(test_dfs)
val_df = pd.concat(val_dfs)

In [22]:
target_col = 'target'
X_cols = list(all_df.columns)
X_cols.remove(target_col)

In [23]:
X_train = train_df[X_cols]
y_train = train_df[[target_col]]

In [24]:
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)

In [25]:
X_train_scaled_df = pd.DataFrame(data=X_train_scaled, columns=X_cols)
X_train_scaled_df.to_parquet('../data/model-ready/water-weather-wind/scaled-X-train.parquet')
y_train.to_parquet('../data/model-ready/water-weather-wind/Y-train.parquet')

In [26]:
X_test = test_df[X_cols]
y_test = test_df[[target_col]]

X_test_scaled = scaler.transform(X_test)
X_test_scaled_df = pd.DataFrame(data=X_test_scaled, columns=X_cols)
X_test_scaled_df.to_parquet('../data/model-ready/water-weather-wind/scaled-X-test.parquet')
y_test.to_parquet('../data/model-ready/water-weather-wind/Y-test.parquet')

In [27]:
X_val = val_df[X_cols]
y_val = val_df[[target_col]]

X_val_scaled = scaler.transform(X_val)
X_val_scaled_df = pd.DataFrame(data=X_val_scaled, columns=X_cols)
X_val_scaled_df.to_parquet('../data/model-ready/water-weather-wind/scaled-X-val.parquet')
y_val.to_parquet('../data/model-ready/water-weather-wind/Y-val.parquet')

In [28]:
joblib.dump(scaler, '../data/model-ready/water-weather-wind/minmax_scaler.joblib')

['../data/model-ready/water-weather-wind/minmax_scaler.joblib']